# sinc RE

In [ ]:
# =====پیش‌نیازها =====
!pip install TorchCRF pytorch-crf
!pip install --quiet transformers datasets seqeval torch evaluate torchcrf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/s eta 0:00:00


In [ ]:
# =====کتابخانه‌ها =====
import os, json
import torch
import numpy as np
import pandas as pd
from collections import Counter
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint
from torchcrf import CRF
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import List, Dict, Any
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

MODEL_NAME = "allenai/scibert_scivocab_cased"
torch.backends.cuda.matmul.allow_tf32 = True
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def read_iob_file(file_path):
    sentences, labels = [], []
    tokens, tags = [], []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                parts = line.split()
                tok, tag = parts[0], parts[-1]
                tokens.append(tok)
                tags.append(tag)
        if tokens:
            sentences.append(tokens)
            labels.append(tags)
    return sentences, labels

# مسیر فایل‌ها
train_file = "/content/drive/MyDrive/poroje/data/science_train_iob.txt"
test_file = "/content/drive/MyDrive/poroje/data/science_test_iob.txt"
train_sents, train_tags = read_iob_file(train_file)
test_sents, test_tags = read_iob_file(test_file)

unique_labels = sorted({t for seq in train_tags + test_tags for t in seq})
combined_labels = ["O"] + [l for l in unique_labels if l != "O"]
label2id = {l: i for i, l in enumerate(combined_labels)}
id2label = {i: l for l, i in label2id.items()}

flat_tags = [t for seq in train_tags for t in seq]
counts = Counter(flat_tags)
total = sum(counts.values())
num_lbls = len(combined_labels)
class_weights = {
    lbl: total / (num_lbls * counts.get(lbl, 1))
    for lbl in combined_labels
}
weights_tensor = torch.tensor([class_weights[lbl] for lbl in combined_labels], dtype=torch.float)

for cls in [
    "B-MATERIAL", "I-MATERIAL",
    "B-OTHERSCIENTIFICTERM", "I-OTHERSCIENTIFICTERM",
    "B-TASK", "I-TASK"
]:
    if cls in label2id:
        weights_tensor[label2id[cls]] *= 2.0

train_examples = [
    {"tokens": s, "ner_tags": [label2id[t] for t in ts]}
    for s, ts in zip(train_sents, train_tags)
]
eval_examples = [
    {"tokens": s, "ner_tags": [label2id[t] for t in ts]}
    for s, ts in zip(test_sents, test_tags)
]
datasets = DatasetDict({
    "train": Dataset.from_list(train_examples),
    "eval": Dataset.from_list(eval_examples)
})
print(datasets)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
MAX_LEN = 320

def tokenize_and_align_iob(ex):
    enc = tokenizer(
        ex["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        add_special_tokens=False,
        return_offsets_mapping=True
    )
    word_ids = enc.word_ids()
    enc.pop("offset_mapping")
    labels_aln, token_mask = [], []
    prev = None
    for wi in word_ids:
        if wi is None:
            labels_aln.append(label2id["O"])
            token_mask.append(0)
        elif wi != prev:
            labels_aln.append(ex["ner_tags"][wi])
            token_mask.append(1)
        else:
            tag = combined_labels[ex["ner_tags"][wi]]
            if tag.startswith("B-"):
                tag = tag.replace("B-", "I-", 1)
            labels_aln.append(label2id[tag])
            token_mask.append(1)
        prev = wi
    enc["labels"] = labels_aln
    enc["token_mask"] = token_mask
    return enc

tokenized = datasets.map(
    tokenize_and_align_iob,
    batched=False,
    remove_columns=["tokens", "ner_tags"]
)
print(tokenized)

@dataclass
class MyTokenCollator:
    tokenizer: Any
    label_pad_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        labels = [f.pop("labels") for f in features]
        masks = [f.pop("token_mask") for f in features]
        batch = self.tokenizer.pad(features, return_tensors="pt")
        max_len = batch["input_ids"].size(1)

        def pad_to(x, pad_val, L):
            return x + [pad_val] * (L - len(x))

        batch["labels"] = torch.tensor(
            [pad_to(l, self.label_pad_token_id, max_len) for l in labels],
            dtype=torch.long
        )
        batch["token_mask"] = torch.tensor(
            [pad_to(m, 0, max_len) for m in masks],
            dtype=torch.bool
        )
        return batch

data_collator = MyTokenCollator(tokenizer, label_pad_token_id=label2id["O"])

# ===== مدل SciBERT + Transformer + CRF + Boundary Head =====
class BertCRFForTokenClassification(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_labels: int,
        labels_vocab,
        feature_extraction: bool = False,
        class_weights: torch.Tensor = None,
        focal_alpha: float = 2.0,
        focal_gamma: float = 3.0,
        boundary_beta: float = 0.3
    ):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        if feature_extraction:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.dropout1 = nn.Dropout(0.3)
        self.dropout2 = nn.Dropout(0.3)
        self.word_transformer_layer = TransformerEncoderLayer(
            d_model=self.bert.config.hidden_size,
            nhead=8,
            dim_feedforward=self.bert.config.hidden_size * 4,
            batch_first=True
        )
        self.word_transformer = TransformerEncoder(self.word_transformer_layer, num_layers=1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        self.class_weights = class_weights
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.labels_vocab = labels_vocab

        self.boundary_classifier = nn.Linear(self.bert.config.hidden_size, 2)
        self.boundary_beta = boundary_beta
        self.register_buffer(
            "is_B",
            torch.tensor([1 if label.startswith("B-") else 0 for label in self.labels_vocab], dtype=torch.long)
        )
        self._apply_bio_constraints()

    def _apply_bio_constraints(self):
        L = self.labels_vocab

        def typ(lbl):
            return lbl.split("-", 1)[1] if lbl != "O" else ""

        with torch.no_grad():
            for i, li in enumerate(L):
                if li.startswith("I-"):
                    self.crf.start_transitions[i] = -1e4
                for j, lj in enumerate(L):
                    if lj.startswith("I-"):
                        ok_prev = li == f"B-{typ(lj)}" or li == f"I-{typ(lj)}"
                        if not ok_prev:
                            self.crf.transitions[i, j] = -1e4

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None, token_mask=None):
        x = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        ).last_hidden_state
        x = self.dropout1(x)
        valid_mask = token_mask.bool() if token_mask is not None else attention_mask.bool()
        x = self.word_transformer(x, src_key_padding_mask=~valid_mask)
        x = self.dropout2(x)
        emissions = self.classifier(x)
        boundary_logits = self.boundary_classifier(x)

        if labels is not None:
            b, s, n = emissions.size()
            em = emissions.reshape(-1, n)[valid_mask.reshape(-1)]
            lb = labels.reshape(-1)[valid_mask.reshape(-1)]
            ce = F.cross_entropy(
                em,
                lb,
                weight=self.class_weights.to(em.device) if self.class_weights is not None else None,
                reduction="none"
            )
            pt = torch.exp(-ce)
            focal = (self.focal_alpha * (1 - pt) ** self.focal_gamma * ce).mean()
            crf_loss = -self.crf(emissions, labels, mask=valid_mask, reduction="mean")

            boundary_targets = self.is_B[labels]
            bl = boundary_logits[valid_mask]
            bt = boundary_targets[valid_mask]
            boundary_loss = F.cross_entropy(bl, bt) if bt.numel() > 0 else emissions.sum() * 0.0
            total_loss = crf_loss + focal + self.boundary_beta * boundary_loss
            return {"loss": total_loss, "logits": emissions}

        best_paths = self.crf.decode(emissions, mask=valid_mask)
        max_len = emissions.size(1)
        padded = [seq + [0] * (max_len - len(seq)) for seq in best_paths]
        return torch.tensor(padded, device=emissions.device)

# ==== تنظیمات آموزش =====
EPOCHS = 30
CHECKPOINT_DIR = "./temp"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model = BertCRFForTokenClassification(
    MODEL_NAME,
    len(combined_labels),
    labels_vocab=combined_labels,
    feature_extraction=False,
    class_weights=weights_tensor,
    focal_alpha=2.0,
    focal_gamma=3.0,
    boundary_beta=0.3
)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.4,
    logging_dir=os.path.join(CHECKPOINT_DIR, "logs"),
    logging_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=1,
    load_best_model_at_end=False,
    bf16=True,
    tf32=True,
    optim="adamw_torch_fused",
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    do_train=True,
    do_eval=True,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["eval"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

last_ckpt = get_last_checkpoint(CHECKPOINT_DIR)
trainer.train(resume_from_checkpoint=last_ckpt if last_ckpt is not None else None)
eval_results = trainer.evaluate()
print("\n=== Eval (HF; loss-based) ===")
print(eval_results)

eval_loader = DataLoader(
    tokenized["eval"],
    batch_size=8,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=2
)
model.eval()
device = next(model.parameters()).device
all_true, all_pred = [], []

with torch.no_grad():
    for batch in eval_loader:
        labels = batch.pop("labels")
        tmask = batch["token_mask"]
        batch = {k: v.to(device) for k, v in batch.items()}
        decoded_ids = model(**batch).cpu()

        for true_ids, pred_ids, m in zip(labels, decoded_ids, tmask):
            seq_true, seq_pred = [], []
            for t_id, p_id, mk in zip(true_ids, pred_ids, m):
                if mk:
                    seq_true.append(id2label[int(t_id)])
                    seq_pred.append(id2label[int(p_id)])
            all_true.append(seq_true)
            all_pred.append(seq_pred)

print("\n--- Classification Report ---")
print(classification_report(all_true, all_pred, digits=4))
print(f"\nOverall Precision: {precision_score(all_true, all_pred):.4f}")
print(f"Overall Recall:    {recall_score(all_true, all_pred):.4f}")
print(f"Overall F1:        {f1_score(all_true, all_pred):.4f}")

Train sentences: 57263
Test  sentences: 551


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/107039 [00:00<?, ? examples/s]

Map:   0%|          | 0/551 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels', 'token_mask'],
        num_rows: 107039
    })
    eval: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels', 'token_mask'],
        num_rows: 551
    })
})
--- Classification Report

                     precision    recall  f1-score   support
GENERIC                 0.6926    0.7759    0.7319       241
MATERIAL                0.6980    0.6582    0.6775       158
METHOD                  0.7897    0.7163    0.7312       775
METRIC                  0.9342    0.6243    0.7584      1432
OTHERSCIENTIFICTERM     0.6330    0.6635    0.6479       520
TASK                    0.6084    0.8437    0.7070      2259

micro avg               0.6933    0.7412    0.7164      5385
macro avg               0.7260    0.7137    0.7090      5385
weighted avg            0.7299    0.7412    0.7187      5385

Overall Precision: 0.6933
Overall Recall:    0.7412
Over